# 01 - Fundamentos de Clasificación y Regresión Logística

## Curso de Machine Learning: Modelos de Clasificación

En este cuaderno iniciaremos el estudio de los problemas de **clasificacion supervisada**. Analizaremos por que los modelos de regresion lineal fallan cuando se aplican a variables objetivo cualitativas, deduciremos formalmente la funcion sigmoide (logistica) y el concepto de log-odds (logit), exploraremos el dataset de diagnostico de cancer de mama de Wisconsin (`load_breast_cancer`), y ajustaremos un modelo de regresion logistica simple analizando su curva de probabilidad y su umbral de decision.

---

### Contenidos
1. Naturaleza de los Problemas de Clasificacion
2. Por que la Regresion Lineal Falla en Clasificacion
3. La Funcion Sigmoide y la Transformacion Logit
4. Exploracion del Dataset Cancer de Mama (`load_breast_cancer`)
5. Analisis Exploratorio y Separabilidad de Clases
6. Regresion Logistica Simple con una Variable Predictora
7. Visualizacion de la Curva de Probabilidad y Umbral de Decision
8. Conclusiones y Siguientes Pasos


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Configuracion visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Naturaleza de los Problemas de Clasificación

En contraste con la regresion (donde la variable objetivo $y \in \mathbb{R}$ es cuantitativa y continua), en **clasificacion** la variable objetivo pertenece a un conjunto discreto y finito de etiquetas o categorias:

$$y \in \{0, 1\} \quad \text{(Clasificacion Binaria)}$$
$$y \in \{0, 1, 2, \dots, K-1\} \quad \text{(Clasificacion Multiclase)}$$

El objetivo fundamental no es estimar un valor numerico arbitrario, sino modelar la **probabilidad a posteriori** de que una observacion pertenezca a una clase determinada dadas sus caracteristicas:

$$p(x) = P(Y = 1 \mid X = x)$$


## 2. Por qué la Regresión Lineal Falla en Clasificación

Intentar ajustar un modelo lineal ordinario $\hat{y} = \beta_0 + \beta_1 x$ para clasificacion binaria codificando $y \in \{0, 1\}$ presenta dos fallos estructurales graves:

1. **Predicciones fuera del rango probabilistico**: Para valores extremos de $x$, la recta produce valores $\hat{y} < 0$ o $\hat{y} > 1$, los cuales no tienen interpretacion valida como probabilidades.
2. **Sensibilidad severa a valores atipicos (Outliers)**: Observaciones situadas lejos de la zona de transicion (incluso si estan correctamente clasificadas) fuerzan a la recta de minimos cuadrados a rotar, desplazando el umbral de clasificacion y provocando errores graves en muestras cercanas a la frontera.

A continuacion, ilustramos este defecto con un experimento sintetico.


In [ ]:
# Demostracion: Vulnerabilidad de la Regresion Lineal ante Outliers
x_demo = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 25]) # 25 es un outlier extremo
y_demo = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1,  1,  1])

# Modelo lineal ordinario
modelo_lineal = LinearRegression().fit(x_demo.reshape(-1, 1), y_demo)

# Modelo sin el outlier para contrastar
modelo_lineal_sin_outlier = LinearRegression().fit(x_demo[:-1].reshape(-1, 1), y_demo[:-1])

x_plot = np.linspace(0, 26, 300).reshape(-1, 1)
y_pred_con = modelo_lineal.predict(x_plot)
y_pred_sin = modelo_lineal_sin_outlier.predict(x_plot)

plt.figure(figsize=(11, 5))
plt.scatter(x_demo[:-1], y_demo[:-1], color='#2b5c8f', s=60, label='Muestras Normales', zorder=3)
plt.scatter(x_demo[-1], y_demo[-1], color='#d62728', s=100, marker='X', label='Outlier Extremo', zorder=3)

plt.plot(x_plot, y_pred_sin, 'g--', linewidth=2, label='Regresion Lineal (Sin Outlier)')
plt.plot(x_plot, y_pred_con, 'r-', linewidth=2, label='Regresion Lineal (Con Outlier: Frontera Desplazada)')

plt.axhline(0.5, color='gray', linestyle=':', label='Umbral Tipico de Decision (p = 0.5)')
plt.axhline(0.0, color='black', alpha=0.3)
plt.axhline(1.0, color='black', alpha=0.3)
plt.xlabel('Variable Predictora (x)', fontsize=11, fontweight='bold')
plt.ylabel('Valor Predicho ($\\hat{y}$)', fontsize=11, fontweight='bold')
plt.title('Falla de la Regresion Lineal: El Outlier Rota la Recta y Altera la Clasificacion', fontsize=12, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.ylim(-0.2, 1.3)
plt.tight_layout()
plt.show()


## 3. La Función Sigmoide y la Transformación Logit

Para garantizar que las predicciones pertenezcan estrictamente al intervalo $(0, 1)$, mapeamos la combinacion lineal $z = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p$ a traves de la **funcion sigmoide** (o logistica):

$$\sigma(z) = \frac{1}{1 + e^{-z}} = \frac{e^z}{1 + e^z}$$

### Propiedades Matematicas Clave:
- $\lim_{z \to \infty} \sigma(z) = 1$
- $\lim_{z \to -\infty} \sigma(z) = 0$
- Para $z = 0$: $\sigma(0) = 0.5$ (punto critico de decision).
- Su derivada tiene una forma analitica notablemente simple:
  $$\sigma'(z) = \sigma(z)(1 - \sigma(z))$$

### Relacion con los Log-Odds (Logit):
Definiendo $p = \sigma(z) = P(Y = 1 \mid X)$, la razon de probabilidades (odds) de que ocurra el evento frente a que no ocurra es:

$$\text{Odds} = \frac{p}{1 - p} = e^{\beta_0 + \beta_1 x_1 + \dots + \beta_p x_p}$$

Aplicando el logaritmo natural, obtenemos la funcion **logit**:

$$\ln\left(\frac{p}{1 - p}\right) = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p$$

**Interpretacion**: La regresion logistica es un modelo **lineal en el espacio de los log-odds**, aunque su salida probabilistica sea no lineal en el espacio original.


In [ ]:
# Visualizacion grafica de la funcion Sigmoide
z = np.linspace(-7, 7, 300)
sigma = 1 / (1 + np.exp(-z))
derivada = sigma * (1 - sigma)

plt.figure(figsize=(10, 5))
plt.plot(z, sigma, color='#2b5c8f', linewidth=2.5, label=r'Sigmoide: $\sigma(z) = \frac{1}{1 + e^{-z}}$')
plt.plot(z, derivada, color='#d95f02', linestyle='--', linewidth=2, label=r"Derivada: $\sigma'(z) = \sigma(z)(1-\sigma(z))$")

plt.axhline(0.5, color='gray', linestyle=':', alpha=0.7)
plt.axvline(0.0, color='gray', linestyle=':', alpha=0.7)
plt.scatter(0, 0.5, color='red', s=70, zorder=4, label='Punto de Inflexion (z=0, p=0.5)')

plt.title('Mapeo No Lineal de la Funcion Sigmoide', fontsize=13, fontweight='bold')
plt.xlabel('Combinacion Lineal z = beta^T x', fontsize=11, fontweight='bold')
plt.ylabel('Probabilidad Estimada P(Y=1|X)', fontsize=11, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


## 4. Exploración del Dataset Cáncer de Mama (`load_breast_cancer`)

El dataset de Wisconsin (Breast Cancer Diagnostic) describe caracteristicas computadas a partir de imagenes digitalizadas de biopsias por aspiracion con aguja fina (FNA) de masas mamarias.

- **Numero de Muestras**: 569 pacientes.
- **Numero de Caracteristicas**: 30 variables continuas cuantitativas (radio, textura, perimetro, area, suavidad, compacidad, concavidad, puntos concavos, simetria y dimension fractal calculados en valores medios, error estandar y peor caso).
- **Variable Objetivo (`target`)**:
  - `0`: Maligno (Malignant) - Tumor canceroso.
  - `1`: Benigno (Benign) - Tumor no canceroso.


In [ ]:
cancer = load_breast_cancer(as_frame=True)
df_cancer = cancer.frame

print(f"Dimensiones del dataset: {df_cancer.shape}")
print(f"Distribucion de la variable objetivo:")
conteo = df_cancer['target'].value_counts()
print(f"  Clase 1 (Benigno):  {conteo[1]} casos ({conteo[1]/len(df_cancer)*100:.1f}%)")
print(f"  Clase 0 (Maligno): {conteo[0]} casos ({conteo[0]/len(df_cancer)*100:.1f}%)")
print("\nPrimeras 5 observaciones (primeras 6 columnas):")
df_cancer.iloc[:, :6].head()


## 5. Análisis Exploratorio y Separabilidad de Clases

Para construir un primer modelo didactico, seleccionaremos una caracteristica con alta capacidad discriminante para separar tumores malignos de benignos. Examinaremos `mean concave points` (promedio del numero de porciones concavas del contorno celular).


In [ ]:
var_seleccionada = 'mean concave points'

plt.figure(figsize=(10, 5))
sns.histplot(data=df_cancer, x=var_seleccionada, hue='target', bins=30, kde=True,
             palette=['#d62728', '#2b5c8f'], element='step', alpha=0.4)
plt.title(f'Distribucion de "{var_seleccionada}" segun Estado del Tumor', fontsize=12, fontweight='bold')
plt.xlabel('Puntos Concavos Promedio (mean concave points)', fontsize=11, fontweight='bold')
plt.ylabel('Frecuencia', fontsize=11, fontweight='bold')
plt.legend(labels=['Benigno (1)', 'Maligno (0)'], title='Diagnostico', fontsize=10)
plt.tight_layout()
plt.show()


## 6. Regresión Logística Simple con una Variable Predictora

Ajustaremos un modelo de regresion logistica utilizando exclusivamente `mean concave points` para predecir si el tumor es benigno ($y=1$) o maligno ($y=0$).


In [ ]:
X_simple = df_cancer[[var_seleccionada]]
y_simple = df_cancer['target']

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_simple, y_simple, test_size=0.20, random_state=42, stratify=y_simple
)

# Ajuste del modelo logistico
modelo_log_simple = LogisticRegression(random_state=42)
modelo_log_simple.fit(X_train_s, y_train_s)

beta_0 = modelo_log_simple.intercept_[0]
beta_1 = modelo_log_simple.coef_[0][0]

print(f"Intercepto (beta_0): {beta_0:.4f}")
print(f"Coeficiente (beta_1): {beta_1:.4f}")

# Calculo analitico de la frontera de decision (donde p = 0.5 <=> z = 0 <=> beta_0 + beta_1 * x = 0)
x_frontera = -beta_0 / beta_1
print(f"Frontera de decision analitica (p = 0.5): {var_seleccionada} = {x_frontera:.4f}")


## 7. Visualización de la Curva de Probabilidad y Umbral de Decisión

Graficamos los datos reales observados junto con la curva sigmoide generada por el estimador y el umbral de clasificacion en $p = 0.5$.


In [ ]:
x_malla = np.linspace(X_simple[var_seleccionada].min(), X_simple[var_seleccionada].max(), 500).reshape(-1, 1)
probabilidades = modelo_log_simple.predict_proba(x_malla)[:, 1] # Probabilidad de pertenecer a la clase 1 (Benigno)

plt.figure(figsize=(11, 5.5))

# Puntos reales
plt.scatter(X_train_s[var_seleccionada], y_train_s, color='#333333', alpha=0.4,
            label='Datos Reales (0 = Maligno, 1 = Benigno)', s=40)

# Curva de probabilidad estimada
plt.plot(x_malla, probabilidades, color='#1f77b4', linewidth=2.5,
         label='Probabilidad Estimada: P(Y=1 | X)')

# Frontera de decision
plt.axvline(x_frontera, color='#d62728', linestyle='--', linewidth=2,
            label=f'Frontera de Decision (x* = {x_frontera:.4f})')
plt.axhline(0.5, color='gray', linestyle=':', alpha=0.8)

plt.xlabel('Puntos Concavos Promedio (mean concave points)', fontsize=11, fontweight='bold')
plt.ylabel('Probabilidad Estimada de ser Benigno', fontsize=11, fontweight='bold')
plt.title('Curva Sigmoide de Regresion Logistica y Umbral de Decision', fontsize=13, fontweight='bold')
plt.legend(loc='center right', fontsize=10)
plt.tight_layout()
plt.show()

# Evaluacion de precision en conjunto de prueba
y_pred_s = modelo_log_simple.predict(X_test_s)
acc_test = accuracy_score(y_test_s, y_pred_s)
print(f"Exactitud (Accuracy) en conjunto de prueba con una sola variable: {acc_test*100:.2f}%")


## 8. Conclusiones y Siguientes Pasos

### Puntos Clave Aprendidos:
1. **Inadecuacion de OLS para clasificacion**: Produce valores fuera de $[0, 1]$ y es extremadamente sensible a observaciones atipicas.
2. **La Funcion Sigmoide**: Mapea la recta lineal $z = \beta^T x$ hacia un rango valido de probabilidades $(0, 1)$, con una derivada computable de manera analitica directa.
3. **Log-Odds**: El modelo asume linealidad en los log-odds de la razon de probabilidades, proporcionando una base probabilistica solida.
4. **Frontera de Decision**: Para el umbral estandar $p=0.5$, la decision ocurre en $z=0$, lo que define una frontera de separacion lineal en el espacio de caracteristicas.

En el siguiente cuaderno, **02_Funcion_Costo_Optimizacion_y_Frontera_Decision.ipynb**, deduciremos por que el error cuadratico medio (MSE) produce funciones no convexas en clasificacion, formularemos la **Entropia Cruzada Binaria (Log-Loss)** y visualizaremos fronteras de decision bidimensionales con mapas de probabilidad.
